## Lab 4: Deploy to Production - Use AgentCore Runtime with Observability

### Overview

In Lab 3 we scaled our Financial Product Launch Agent by centralizing tools through AgentCore Gateway. Now we'll deploy to AgentCore Runtime for production with full observability.

**Workshop Journey:**
- **Lab 1 (Done):** Create Agent Prototype
- **Lab 2 (Done):** Enhance with Memory
- **Lab 3 (Done):** Scale with Gateway & Identity
- **Lab 4 (Current):** Deploy to Production - AgentCore Runtime
- **Lab 5:** Build User Interface

### Why AgentCore Runtime?

**Current State:** Agent runs locally
**After Lab 4:** Production-ready with:
- Serverless auto-scaling
- Comprehensive observability (traces, metrics, logs)
- Enterprise reliability
- Secure deployment

### Prerequisites
- Python 3.11+
- AWS account with permissions
- Docker installed
- Lab 2 completed (Memory)
- Lab 3 completed (Gateway)

### Step 1: Import Required Libraries

In [ ]:
import os
import sys
import boto3
from boto3.session import Session

# Add current directory to path
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

boto_session = Session()
region = boto_session.region_name

print(f"✅ Region: {region}")

In [ ]:
from lab_helpers.utils import get_ssm_parameter

try:
    memory_id = get_ssm_parameter("/app/productlaunch/agentcore/memory_id")
    print(f"✅ Memory found: {memory_id}")
except:
    print("⚠️ Memory not found - run Lab 2 first")

In [ ]:
try:
    gateway_id = get_ssm_parameter("/app/productlaunch/agentcore/gateway_id")
    print(f"✅ Gateway found: {gateway_id}")
except:
    print("⚠️ Gateway not found - run Lab 3 first")

### Step 2: Preparing Your Agent for AgentCore Runtime

#### Creating the Runtime-Ready Agent

Let's first define the necessary AgentCore Runtime components via Python SDK within our previous local agent implementation.

Observe the `#### AGENTCORE RUNTIME - LINE i ####` comments below to see where is the relevant deployment code added. You'll find 4 such lines that prepare the runtime-ready agent:

1. Import the Runtime App with `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
2. Initialize the App with `app = BedrockAgentCoreApp()`
3. Decorate our invocation function with `@app.entrypoint`
4. Let AgentCore Runtime control the execution with `app.run()`


In [ ]:
%%writefile lab_helpers/lab4_runtime.py
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent
from strands.models import BedrockModel
from lab_helpers.unified_market_research import market_research
from lab_helpers.enhanced_tools import browse_web
from lab_helpers.marketing_tools import create_marketing_poster

# Create single agent with tools
model = BedrockModel(
    model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    temperature=0.3
)

agent = Agent(
    model=model,
    tools=[market_research, browse_web, create_marketing_poster],
    system_prompt="""You are a financial product launch assistant.
    Use market_research for market analysis, browse_web for competitor research,
    and create_marketing_poster for marketing materials."""
)

# Initialize AgentCore Runtime App
app = BedrockAgentCoreApp()

@app.entrypoint
def invoke(payload):
    """AgentCore Runtime entrypoint - single agent"""
    user_input = payload.get("prompt", "")
    response = agent(user_input)
    return response

if __name__ == "__main__":
    app.run()



#### What happens behind the scenes?

When you use `BedrockAgentCoreApp`, it automatically:

- Creates an HTTP server that listens on port 8080
- Implements the required `/invocations` endpoint for processing requests
- Implements the `/ping` endpoint for health checks
- Handles proper content types and response formats
- Manages error handling according to AWS standards


### Step 3: Deploying to AgentCore Runtime

Now let's deploy our agent to AgentCore Runtime using the [AgentCore Starter Toolkit](https://github.com/aws/bedrock-agentcore-starter-toolkit).

#### Configure the Secure Runtime Deployment (AgentCore Runtime + AgentCore Identity)

First we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we will create and a requirements file. We will also configure the identity authorization using an Amazon Cognito user pool and we will configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code

<div style="text-align:left"> 
    <img src="images/configure.png" width="75%"/> 
</div>

**Note**: The Cognito access_token is valid for 2 hours only. If the access_token expires you can vend another access_token by using the `reauthenticate_user` method.


In [ ]:
from lab_helpers.utils import get_or_create_cognito_pool, reauthenticate_user

print("Setting up Cognito user pool...")
cognito_config = get_or_create_cognito_pool()
print(f"✅ Cognito configured: {cognito_config.get('user_pool_id')}")

In [ ]:
from lab_helpers.utils import create_agentcore_runtime_execution_role

execution_role_arn = create_agentcore_runtime_execution_role()
print(f"✅ Execution role: {execution_role_arn}")

## Configure Runtime

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="lab_helpers/lab4_runtime.py",
    execution_role=execution_role_arn,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="productlaunchagent",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "allowedClients": [cognito_config.get("client_id")],
            "discoveryUrl": cognito_config.get("discovery_url"),
        }
    },
)

print("✅ Runtime configured")

#### Launch the Agent

Now let's launch our agent to AgentCore Runtime. This will create an AWS CodeBuild pipeline, the Amazon ECR repository and the AgentCore Runtime components.

<div style="text-align:left"> 
    <img src="images/launch.png" width="100%"/> 
</div>

In [ ]:
 #Open a Terminal and run the command - rm .bedrock_agentcore.yaml    if the below cell fails

In [ ]:
from lab_helpers.utils import put_ssm_parameter

launch_result = agentcore_runtime.launch()
print(f"✅ Launch completed: {launch_result.agent_arn}")

put_ssm_parameter("/app/productlaunch/agentcore/runtime_arn", launch_result.agent_arn)

#### Check Deployment Status

Let's wait for the deployment to complete:


In [ ]:
import time

# Wait for the agent to be ready
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]

end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    print(f"Waiting for deployment... Current status: {status}")
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]

print(f"Final status: {status}")

### Step 4: Invoking Your Deployed Agent

Now that our agent is deployed and ready, let's test it with some queries. We invoke the agent with the right authorization token type. In out case it'll be Cognito access token. Copy the access token from the cell above


#### Using the AgentCore Starter Toolkit

We can validate that the agent works using the AgentCore Starter Toolkit for invocation. The starter toolkit can automatically create a session id for us to query our agent. Alternatively, you can also pass the session id as a parameter during invocation. For demonstration purpose, we will create our own session id.

In [ ]:
# Test the agent with a financial product query
import uuid
from IPython.display import Markdown, display

# Create new session for marketing campaign
session_id = uuid.uuid4()
user_query = "Research the current auto loan market and help me create a competitive product for millennials"

bearer_token = reauthenticate_user(
    cognito_config.get("client_id"), 
    cognito_config.get("client_secret")
)

response = agentcore_runtime.invoke(
    {"prompt": user_query}, 
    bearer_token=bearer_token,
    session_id=str(session_id)
)
response

### Test 2: Create Marketing Campaign

Test the marketing poster generation capability.

In [ ]:
import uuid
from IPython.display import Markdown, display

# Create new session for marketing campaign
session_id2 = uuid.uuid4()

user_query = "Create a marketing poster for a new credit card with 0% APR for 12 months targeting young professionals"

response = agentcore_runtime.invoke(
    {"prompt": user_query},
    bearer_token=bearer_token,
    session_id=str(session_id2),
)

response.get('response', response)

### Test 3: Competitive Analysis

Test web browsing for competitor research.

In [ ]:
# Create new session for competitive analysis
session_id3 = uuid.uuid4()

user_query = "Browse bankrate.com and analyze their mortgage rate offerings compared to our planned 6.5% 30-year fixed rate"

response = agentcore_runtime.invoke(
    {"prompt": user_query},
    bearer_token=bearer_token,
    session_id=str(session_id3),
)

response.get('response', response)

### Test 4: Memory Persistence

Test that the agent remembers context from previous session.

In [ ]:
# Use the first session ID to test memory
user_query = "What product did we discuss earlier?"

response = agentcore_runtime.invoke(
    {"prompt": user_query},
    bearer_token=bearer_token,
    session_id=str(session_id),  # Same session as first test
)

response.get('response', response)

#### Invoking the agent with a new user
In the example below we have not mentioned the Iphone device in the second query, but our agent still has the context of it. This is due to the AgentCore Runtime session continuity. The agent won't know the context for a new user.

In [ ]:
# Creating a new session ID for demonstrating new product manager
session_id2 = uuid.uuid4()

user_query = "What are the current mortgage rates in the market?"
response = agentcore_runtime.invoke(
    {"prompt": user_query}, 
    bearer_token=bearer_token,
    session_id=str(session_id2)
)
response.get('response', response)

In this case our agent does not have the context anymore and needs more information. 

And it is all it takes to have a secure and scalable endpoint for our Agent with no need to manage all the underlying infrastructure!

### Step 5: AgentCore Observability

[AgentCore Observability](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html) provides monitoring and tracing capabilities for AI agents using Amazon OpenTelemetry Python Instrumentation and Amazon CloudWatch GenAI Observability.

#### Agents

Default AgentCore Runtime configuration allows for logging our agent's traces on CloudWatch by means of **AgentCore Observability**. These traces can be seen on the AWS CloudWatch GenAI Observability dashboard. Navigate to Cloudwatch &rarr; GenAI Observability &rarr; Bedrock AgentCore.

![Agents Overview on CloudWatch](images/obs0.png)

Agent Details: 

![Agents Overview on CloudWatch](images/obs1.png)

Runtime Details: 

![Agents Overview on CloudWatch](images/obs2.png)

Throtthling Details:

![Agents Overview on CloudWatch](images/obs3.png)

#### Sessions

The Sessions view shows the list of all the sessions associated with all agents in your account.

![sessions](images/obs4.png)

#### Traces

Trace view lists all traces from your agents in this account. To work with traces:

- Choose Filter traces to search for specific traces.
- Sort by column name to organize results.
- Under Actions, select Logs Insights to refine your search by querying across your log and span data or select Export selected traces to export.

![traces](images/obs5.png)


### Congratulations! 🎉

You have successfully completed **Lab 4: Deploy to Production - Use AgentCore Runtime with Observability!**

Here is what you accomplished:

##### Production-Ready Deployment:

- Prepared your agent for production with minimal code changes (only 4 lines added)
- Validated proper session isolation between different product managers
- Confirmed session continuity + memory persistence and context awareness per session

##### Enterprise-Grade Security & Identity:

- Implemented secure authentication using Cognito integration with JWT tokens
- Configured proper IAM roles and execution permissions for production workloads
- Established identity-based access control for secure agent invocation

##### Comprehensive Observability:

- Enabled AgentCore Observability for full request tracing across all product manager sessions
- Configured CloudWatch GenAI Observability dashboard monitoring

##### Current Limitations (We'll fix these next!):

- **Developer Focused Interaction** - Agent accessible via SDK/API calls but no user-friendly web interface
- **Manual Session Management** - Requires programmatic session creation rather than intuitive user experience

##### Next Up [Lab 5: Build User Interface →](lab-05-frontend.ipynb)
In Lab 5, you'll complete the product manager experience by building a user-friendly interface !! Lets go !!
